In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
# load all the content of .env file into the environment variables

if os.environ['GROQ_API_KEY']:
    print("API_KEY is set")
else:
    print("API_KEY is not set")

API_KEY is set


In [8]:
# from langchain_groq import 
# from langchain_openai import ChatOpenAI

# for groq
from langchain_groq import ChatGroq

In [11]:

#  llm=ChatOpenAI(model="gpt-5-nano", temperature=0)
# for groq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [14]:
response=llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


# RAG implementation with our own text data

#### step 1 : Preparing Document for own text.

In [15]:
from langchain_core.documents import Document
my_text=""" contextual AI is an enterprise software company[1] based in Mountain View, California. It develops a platform for building[2] specialized Retrieval-Augmented Generation (RAG) agents for enterprise use.[3] The company was founded in 2023 by Douwe Kiela and Amanpreet Singh, both former AI researchers at Facebook AI Research (FAIR)[4] and Hugging Face.[5] Douwe Kiela previously led the Meta research team that introduced the Retrieval-Augmented Generation (RAG) approach in 2020.[6][7][8]

Contextual AI focuses on enterprise generative AI applications using RAG 2.0 technology,[9] with deployments primarily in the technology, banking, finance and media sectors.[10]



"""

docs = Document(page_content=my_text, metadata={"source": "my_text"})
docs

Document(metadata={'source': 'my_text'}, page_content=' contextual AI is an enterprise software company[1] based in Mountain View, California. It develops a platform for building[2] specialized Retrieval-Augmented Generation (RAG) agents for enterprise use.[3] The company was founded in 2023 by Douwe Kiela and Amanpreet Singh, both former AI researchers at Facebook AI Research (FAIR)[4] and Hugging Face.[5] Douwe Kiela previously led the Meta research team that introduced the Retrieval-Augmented Generation (RAG) approach in 2020.[6][7][8]\n\nContextual AI focuses on enterprise generative AI applications using RAG 2.0 technology,[9] with deployments primarily in the technology, banking, finance and media sectors.[10]\n\n\n\n')

#### step 2 : Splitting the Document

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents([docs])
chunks

[Document(metadata={'source': 'my_text'}, page_content='contextual AI is an enterprise software company[1] based in Mountain View, California. It develops a platform for building[2] specialized Retrieval-Augmented Generation (RAG) agents for enterprise use.[3] The company was founded in 2023 by Douwe Kiela and Amanpreet Singh, both former AI researchers at Facebook AI Research (FAIR)[4] and Hugging Face.[5] Douwe Kiela previously led the Meta research team that introduced the Retrieval-Augmented Generation (RAG) approach in 2020.[6][7][8]'),
 Document(metadata={'source': 'my_text'}, page_content='Contextual AI focuses on enterprise generative AI applications using RAG 2.0 technology,[9] with deployments primarily in the technology, banking, finance and media sectors.[10]')]

#### step 3 : Create Embeddings for Chunks

In [18]:
# from langchain_core import OpenAIEmbeddings

# free
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2" 
)

/Users/iwasbinod/Desktop/rag_binod/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6013.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
embeddings.embed_query("What is the capital of France?")

[0.08204811811447144,
 0.03605550527572632,
 -0.0038928694557398558,
 -0.004881022963672876,
 0.025651102885603905,
 -0.05714346840977669,
 0.012191594578325748,
 0.004678945988416672,
 0.03494986891746521,
 -0.02242191694676876,
 -0.008005285635590553,
 -0.10935352742671967,
 0.022724756971001625,
 -0.029320847243070602,
 -0.04352204501628876,
 -0.1202411875128746,
 -0.0008486209553666413,
 -0.018150117248296738,
 0.056129567325115204,
 0.003085286123678088,
 0.0023363730870187283,
 -0.01683926023542881,
 0.06362468749284744,
 -0.023660236969590187,
 0.03149350360035896,
 -0.03479794040322304,
 -0.020548827946186066,
 -0.0027909704949706793,
 -0.011038015596568584,
 -0.03612672537565231,
 0.05414107069373131,
 -0.03661713749170303,
 -0.025008678436279297,
 -0.03817041590809822,
 -0.04960361868143082,
 -0.015148117206990719,
 0.021315021440386772,
 -0.01274044718593359,
 0.07670088857412338,
 0.04435575753450394,
 -0.010834887623786926,
 -0.029760003089904785,
 -0.01697046123445034,
 -

#### step 4 : Create and Store Embeddings in Vector Store.

In [20]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
    )

#### step 5 : Semantic Search

In [25]:
vectorstore.similarity_search("RAG",k=3)

[Document(metadata={'source': 'my_text'}, page_content='Contextual AI focuses on enterprise generative AI applications using RAG 2.0 technology,[9] with deployments primarily in the technology, banking, finance and media sectors.[10]'),
 Document(metadata={'source': 'my_text'}, page_content='contextual AI is an enterprise software company[1] based in Mountain View, California. It develops a platform for building[2] specialized Retrieval-Augmented Generation (RAG) agents for enterprise use.[3] The company was founded in 2023 by Douwe Kiela and Amanpreet Singh, both former AI researchers at Facebook AI Research (FAIR)[4] and Hugging Face.[5] Douwe Kiela previously led the Meta research team that introduced the Retrieval-Augmented Generation (RAG) approach in 2020.[6][7][8]')]

Talk to LLM

In [26]:
context = vectorstore.similarity_search("RAG",k=3)  

In [28]:
response=llm.invoke(f"What is RAG? You can answer using the following context: {context}")
print(response.content)

Based on the provided context, RAG stands for Retrieval-Augmented Generation. It is a technology used in enterprise generative AI applications, particularly in the technology, banking, finance, and media sectors.
